# Task 4 on Kaggle (face-to-sketch cGAN)
Settings: Accelerator = GPU, Internet = ON, secret `GH_TOKEN` attached. Use **Save Version -> Save & Run All (Commit)**.

In [ ]:
import base64, os, subprocess
BRANCH = 'dev'
REPO = 'AfnanRizwan1/GenAI-A1'
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
except Exception as e:
    TOKEN = None
    print(f'GH_TOKEN secret not available ({type(e).__name__}). Private repo: right panel -> Add-ons -> Secrets -> switch GH_TOKEN on for THIS notebook, then restart.')

def git(*args):
    """Run git; the token is sent as a one-off header, never stored in .git/config or printed."""
    cmd, secrets = ['git'], []
    if TOKEN:
        b64 = base64.b64encode(f'x-access-token:{TOKEN}'.encode()).decode()
        cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {b64}']
        secrets = [TOKEN, b64]
    r = subprocess.run(cmd + list(args), capture_output=True, text=True)
    out = r.stdout + r.stderr
    for s in secrets:
        out = out.replace(s, '***')
    if r.returncode:
        raise RuntimeError(f'git {args[0]} failed:\n{out}')
    return out

os.chdir('/kaggle/working')
if not os.path.exists('GenAI-A1'):
    git('clone', '-b', BRANCH, f'https://github.com/{REPO}.git')
os.chdir('GenAI-A1')
git('pull', 'origin', BRANCH)
print(git('log', '--oneline', '-3'))

In [ ]:
!pip install -q pytorch-msssim optuna mlflow onnx onnxruntime opencv-python-headless scikit-learn gdown
!nvidia-smi | head -12
!python -m pytest tests/test_cgan.py -q 2>&1 | tail -3

In [ ]:
# FS2K from the official Google Drive link (see github.com/DengPingFan/FS2K). Skip if you attached it as a Kaggle dataset.
import os, glob
if not glob.glob('data/fs2k/**/anno_train.json', recursive=True):
    os.makedirs('data/fs2k', exist_ok=True)
    os.system('gdown 1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp -O data/fs2k.zip && unzip -q data/fs2k.zip -d data/fs2k')
print(glob.glob('data/fs2k/**/anno_*.json', recursive=True))

In [ ]:
# builds the 128x128 cache and prints the split sizes (train / val / official test)
!python -c "from src.data.fs2k import load_fs2k; d = load_fs2k('data/fs2k'); print({k: v['photo'].shape for k, v in d.items()}); import numpy as np; print({k: np.bincount(v['style']).tolist() for k, v in d.items()})"

In [ ]:
!GPU=0 TRIALS=20 TRIAL_EPOCHS=15 FINAL_EPOCHS=150 FS2K_ROOT=data/fs2k bash scripts/run_t4.sh